# 14 Class 2 vs 3: which features, which model?

Single features, feature-count curve, and a four-model comparison on the 15 shape features, then a replication on fresh fold sets. Rule: a model beats another only if its mean AUC is more than 0.03 higher and it wins in all 5 repeats; otherwise it is a tie and the simpler model stays.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_allclass.csv")
data = feat[feat["y"].isin([2, 3])].reset_index(drop=True)          # Kaggle classes 2 and 3, development set only
assert set(data["segment"]) <= set(table["segment"])

models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.svm")     # SVC(probability=True) warning, this notebook only

## Single features (logistic regression, one feature at a time)

In [2]:
single = {}
for feature in FEATURES:
    r = cross_validate_segments(models["logistic regression"], data, [feature], table, positive=2)
    single[feature] = (r["auc_fold"].mean(), r["auc_fold"].std())
single = pd.DataFrame(single, index=["auc", "sd"]).T.sort_values("auc", ascending=False)
single["family"] = ["size" if f in SIZE_FEATURES else "shape" for f in single.index]
display(single.round(3))

,auc,sd,family
hjorth_complexity,0.697,0.010,shape
rel_alpha,0.641,0.024,shape
logratio_slow_fast,0.635,0.020,shape
skewness,0.621,0.012,shape
logpow_alpha,0.616,0.018,size
rel_delta,0.610,0.034,shape
logpow_theta,0.606,0.025,size
spectral_centroid,0.605,0.014,shape
logpow_beta,0.592,0.011,size
logratio_theta_alpha,0.580,0.022,shape


## Best k features, chosen inside each training fold

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif

curve = {}
for k in (1, 2, 3, 5, 8, 12, 23):
    make_model = lambda k=k: make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k), LogisticRegression(max_iter=1000))
    r = cross_validate_segments(make_model, data, FEATURES, table, positive=2)
    curve[k] = (r["auc_fold"].mean(), r["auc_fold"].std())
display(pd.DataFrame(curve, index=["auc", "sd"]).T.round(3))

,auc,sd
1,0.660,0.015
2,0.649,0.004
3,0.636,0.010
5,0.622,0.015
8,0.639,0.020
12,0.623,0.026
23,0.708,0.025


## Four models on the 15 shape features

In [4]:
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier

family = {
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM (RBF)": lambda: make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
    "gradient boosting": lambda: HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
}

def compare(d, tab):
    results = {name: cross_validate_segments(mk, d, SHAPE_FEATURES, tab, positive=2) for name, mk in family.items()}
    auc = pd.DataFrame({name: r["auc_fold"] for name, r in results.items()})          # one row per repeat
    diff = auc.sub(auc["logistic regression"], axis=0)                                 # difference from logistic, per repeat
    summary = pd.DataFrame({"mean AUC": auc.mean(), "sd over repeats": auc.std(),
                            "mean diff vs logistic": diff.mean(), "repeats better than logistic (of 5)": (diff > 0).sum()})
    return results, summary

results, summary = compare(data, table)
display(summary.round(3))
display(pd.DataFrame({name: r[["pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"]].mean()
                      for name, r in results.items()}).T.round(3))

,mean AUC,sd over repeats,mean diff vs logistic,repeats better than logistic (of 5)
logistic regression,0.749,0.018,0.000,0
SVM (RBF),0.793,0.015,0.044,5
random forest,0.779,0.024,0.029,5
gradient boosting,0.712,0.024,-0.038,1


,pr_auc_fold,sensitivity,specificity,precision,f1,balanced_accuracy
logistic regression,0.821,0.640,0.768,0.735,0.684,0.704
SVM (RBF),0.828,0.725,0.760,0.751,0.738,0.742
random forest,0.808,0.695,0.765,0.747,0.720,0.730
gradient boosting,0.754,0.668,0.658,0.661,0.664,0.662


## Replication on fresh fold sets

Same 160 segments and same twin groups, but new fold shuffles (seeds 1000, 2000, 3000). If the SVM lead over logistic is real, it should show up here too. Rule: real if the mean AUC gain is above 0.03 on at least 2 of 3 fold sets.

In [5]:
from src.cv import make_group_fold_table

group_of = data.set_index("segment")["group"]
fresh = {}
for seed in (1000, 2000, 3000):
    new_table = make_group_fold_table(data, group_of, seed=seed)
    _, s = compare(data, new_table)
    fresh[seed] = s
    print(f"seed {seed}")
    display(s.round(3))

gain = pd.DataFrame({seed: s["mean diff vs logistic"] for seed, s in fresh.items()})
print("mean AUC difference from logistic, per fold set")
display(gain.round(3))
for name in ["SVM (RBF)", "random forest", "gradient boosting"]:
    n = int((gain.loc[name] > 0.03).sum())
    print(f"{name:18s} above +0.03 on {n} of 3 fold sets -> {'real' if n >= 2 else 'not shown'}")

seed 1000


,mean AUC,sd over repeats,mean diff vs logistic,repeats better than logistic (of 5)
logistic regression,0.773,0.023,0.000,0
SVM (RBF),0.792,0.012,0.019,4
random forest,0.778,0.030,0.005,3
gradient boosting,0.697,0.026,-0.076,0


seed 2000


,mean AUC,sd over repeats,mean diff vs logistic,repeats better than logistic (of 5)
logistic regression,0.761,0.010,0.000,0
SVM (RBF),0.772,0.019,0.012,4
random forest,0.756,0.013,-0.005,2
gradient boosting,0.707,0.014,-0.053,0


seed 3000


,mean AUC,sd over repeats,mean diff vs logistic,repeats better than logistic (of 5)
logistic regression,0.761,0.022,0.000,0
SVM (RBF),0.772,0.018,0.011,3
random forest,0.759,0.018,-0.001,1
gradient boosting,0.706,0.020,-0.054,0


mean AUC difference from logistic, per fold set


,1000,2000,3000
logistic regression,0.000,0.000,0.000
SVM (RBF),0.019,0.012,0.011
random forest,0.005,-0.005,-0.001
gradient boosting,-0.076,-0.053,-0.054


SVM (RBF)          above +0.03 on 0 of 3 fold sets -> not shown
random forest      above +0.03 on 0 of 3 fold sets -> not shown
gradient boosting  above +0.03 on 0 of 3 fold sets -> not shown


## Findings

- **Single features:** hjorth_complexity is the best by far (0.697). The next are 0.62 to 0.64. spectral_entropy scores 0.335, well below chance, meaning it points the wrong way when the model learns on the other folds. Treat that as noise, not a finding.
- **Feature count:** best k = 1, 2, 3, 5, 8, 12, 23 gives 0.660, 0.649, 0.636, 0.622, 0.639, 0.623, 0.708. Picking a few features does not beat using all of them.
- **Four models on shape features (saved folds):** logistic 0.749, SVM 0.793, forest 0.779, gradient boosting 0.712. On these folds the SVM beats logistic by the rule (+0.044, 5 of 5 repeats).
- **Replication on fresh folds:** the SVM lead shrinks to +0.019, +0.012, +0.011 (seeds 1000, 2000, 3000). It never reaches 0.03, so it is **not** confirmed. The saved folds were a lucky draw for the SVM. Logistic and SVM are a tie, and the simpler logistic stays. Forest is also a tie. Gradient boosting is clearly worse (-0.05 to -0.08).

## Caveats

- The model ranking is within the noise of re-making folds. Do not read it as "SVM is better".
- Same patient-overlap limit as before. Scores are upper bounds.
- Gradient boosting was left at defaults, so its weakness may be partly tuning.